# Day 37：里程碑项目 2 Part 2 — 全栈企业级知识库问答系统 (`v0.3.0-day37`)

本 Notebook 使用 `fastapi.testclient.TestClient` 在进程内完整演示：
1. 使用隔离的临时 SQLite 数据库初始化 `create_app()`
2. 通过 `POST /api/v1/kb/upload` 上传企业制度文档并自动分块索引
3. 通过 `POST /api/v1/kb/chat/stream` 发起多轮 SSE 流式问答并解析 `token`、`citations`、`done` 事件
4. 通过 `GET /api/v1/kb/sessions/{session_id}/history` 验证多轮对话与引用卡片持久化恢复

In [1]:
import json
import sys
import tempfile
from pathlib import Path

from fastapi.testclient import TestClient

# 定位项目根目录与 day_37 迭代目录
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'project_iterations' / 'day_37').exists():
    REPO_ROOT = REPO_ROOT.parent.parent.parent
DAY_DIR = REPO_ROOT / 'project_iterations' / 'day_37'
if str(DAY_DIR) not in sys.path:
    sys.path.insert(0, str(DAY_DIR))

from zhishu_agent.kb_web_app import create_app

print('Day 37 模块加载成功:', DAY_DIR)

Day 37 模块加载成功: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_37


/home/antigravity/ab6539763/-AI-Agent/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [2]:
tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / 'demo_kb.db'
app = create_app(db_path=db_path, frontend_dir=DAY_DIR / 'frontend')
client = TestClient(app)

health = client.get('/api/v1/health').json()
print('健康检查:', health)

# 上传两份企业知识库文档
docs = [
    {
        'filename': 'hr_leave_policy.md',
        'content': '星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。病假需提交三甲医院证明。',
    },
    {
        'filename': 'travel_reimbursement.md',
        'content': '差旅报销标准：一线城市酒店住宿报销上限每晚600元，二线城市每晚450元，需提供电子发票。',
    },
]
for doc in docs:
    res = client.post('/api/v1/kb/upload', json=doc)
    assert res.status_code == 200
    print('已上传:', res.json()['document'])


健康检查: {'status': 'ok', 'version': 'v0.3.0-day37', 'db_path': '/tmp/tmpynavdgxe/demo_kb.db', 'document_count': 0}
已上传: {'doc_id': '320efc984030', 'filename': 'hr_leave_policy.md', 'chunk_count': 1, 'char_count': 49, 'created_at': '2026-11-13T06:35:36.730616+00:00'}
已上传: {'doc_id': 'c3e96c56e952', 'filename': 'travel_reimbursement.md', 'chunk_count': 1, 'char_count': 45, 'created_at': '2026-11-13T06:35:36.740450+00:00'}


In [3]:
def parse_sse(raw_text: str) -> list[dict]:
    events = []
    for block in raw_text.split('\n\n'):
        line = block.strip()
        if line.startswith('data: '):
            events.append(json.loads(line[6:]))
    return events

session_id = 'nb-demo-session'
with client.stream(
    'POST',
    '/api/v1/kb/chat/stream',
    json={'session_id': session_id, 'message': '一线城市酒店住宿报销上限是多少？', 'top_k': 2, 'delay_ms': 0},
) as resp:
    assert resp.status_code == 200
    events = parse_sse(''.join(resp.iter_text()))

tokens = [e['content'] for e in events if e['type'] == 'token']
citations = next(e['citations'] for e in events if e['type'] == 'citations')
done_evt = next(e for e in events if e['type'] == 'done')

print('Token 块数:', len(tokens))
print('完整回答:\n' + ''.join(tokens))
print('引用卡片:', json.dumps(citations, ensure_ascii=False, indent=2))
print('完成事件:', done_evt)

Token 块数: 32
完整回答:
【智枢知识库 · 第 1 轮解答】
针对您的问题「一线城市酒店住宿报销上限是多少？」，基于企业知识库检索结果为您整理如下：
[1] 依据《travel_reimbursement.md》（片段 c3e96c56e952-chunk-01）：差旅报销标准：一线城市酒店住宿报销上限每晚600元，二线城市每晚450元，需提供电子发票。
（点击下方引用卡片可查看对应文档原文与匹配度详情。）
引用卡片: [
  {
    "doc_id": "c3e96c56e952",
    "filename": "travel_reimbursement.md",
    "chunk_id": "c3e96c56e952-chunk-01",
    "score": 0.99,
    "snippet": "差旅报销标准：一线城市酒店住宿报销上限每晚600元，二线城市每晚450元，需提供电子发票。"
  }
]
完成事件: {'type': 'done', 'message_id': 2, 'session_id': 'nb-demo-session'}


In [4]:
# 第二轮追问与 SQLite 历史恢复验证
r2 = client.post(
    '/api/v1/kb/chat/stream',
    json={'session_id': session_id, 'message': '那二线城市呢？', 'top_k': 2, 'delay_ms': 0},
)
assert r2.status_code == 200

history = client.get(f'/api/v1/kb/sessions/{session_id}/history').json()['messages']
print('历史消息总数:', len(history))
for msg in history:
    print(f"[{msg['role']}] {msg['content'][:45]}... | citations={len(msg['citations'])}")

tmp_dir.cleanup()

历史消息总数: 4
[user] 一线城市酒店住宿报销上限是多少？... | citations=0
[assistant] 【智枢知识库 · 第 1 轮解答】
针对您的问题「一线城市酒店住宿报销上限是多少？」，基于... | citations=1
[user] 那二线城市呢？... | citations=0
[assistant] 【智枢知识库 · 第 2 轮解答】
针对您的问题「那二线城市呢？」，基于企业知识库检索结果... | citations=1
